# Raw Snapshot Source Check (Redis / NFS / Local)

用途：
- 读取指定日期的 `NFS raw snapshot`、`本地 raw snapshot`、`Redis snapshot`
- 统计每个来源的时间戳范围（min/max）、日期分布、phase 分布
- 检查 DB 日频表是否包含分钟级时间字段

In [15]:
from pathlib import Path
from datetime import datetime
import sys
import json5
import pandas as pd

ROOT = Path(r"C:/Users/BaiYang/CBOND_ON")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from cbond_on.live.redis_snapshot import RedisConfig, SnapshotRedisClient
from cbond_on.data.io import read_table_range


In [16]:
# ===== 输入参数（先改这里） =====
DATE_STR = "2026-02-06"
DATE = pd.to_datetime(DATE_STR).date()

# Redis读取模式：True=按日期范围拉取；False=按latest拉取后再按日期过滤
REDIS_USE_RANGE = True
REDIS_SYMBOL_LIMIT = 80
REDIS_LATEST_LIMIT = 5000
SHOW_HEAD = 5

paths_cfg = json5.loads((ROOT / "cbond_on/config/paths_config.json5").read_text(encoding="utf-8"))
raw_cfg = json5.loads((ROOT / "cbond_on/config/raw_data_config.json5").read_text(encoding="utf-8"))
live_cfg = json5.loads((ROOT / "cbond_on/config/live_config.json5").read_text(encoding="utf-8"))

raw_root = Path(paths_cfg["raw_data_root"])
local_raw_path = raw_root / "snapshot" / "cbond" / "raw_data" / DATE.strftime("%Y-%m") / f"{DATE:%Y%m%d}.parquet"

nfs_cfg = raw_cfg.get("nfs", {})
nfs_root = Path(str(nfs_cfg.get("nfs_root", "")))
nfs_base_dir = Path(str(nfs_cfg.get("base_dir", "snapshot/cbond/raw_data")).strip("/\\\\"))
nfs_raw_path = nfs_root / nfs_base_dir / DATE.strftime("%Y-%m") / f"{DATE:%Y%m%d}.parquet"

print("DATE:", DATE)
print("local_raw_path:", local_raw_path)
print("nfs_raw_path:", nfs_raw_path)


DATE: 2026-02-06
local_raw_path: D:\cbond_on\raw_data\snapshot\cbond\raw_data\2026-02\20260206.parquet
nfs_raw_path: \\nfs\999.999@10.1.30.100\data\yinhe-data\snapshot\cbond\raw_data\2026-02\20260206.parquet


In [17]:
def _normalize_trade_time(df: pd.DataFrame) -> pd.DataFrame:
    if df is None or df.empty:
        return pd.DataFrame()
    out = df.copy()
    if "trade_time" in out.columns:
        col = "trade_time"
    elif "timestamp" in out.columns:
        col = "timestamp"
    elif "ts" in out.columns:
        col = "ts"
    else:
        out["trade_time"] = pd.NaT
        return out

    if pd.api.types.is_numeric_dtype(out[col]):
        s = pd.to_numeric(out[col], errors="coerce")
        sample = float(s.dropna().abs().median()) if not s.dropna().empty else 0.0
        unit = "ms" if sample >= 1e12 else "s"
        out["trade_time"] = pd.to_datetime(s, unit=unit, errors="coerce")
    else:
        out["trade_time"] = pd.to_datetime(out[col], errors="coerce")
    return out


def summarize_snapshot(df: pd.DataFrame, name: str, expected_day=None):
    print(f"\
===== {name} =====")
    if df is None or df.empty:
        print("empty")
        return {"source": name, "rows": 0}

    dfn = _normalize_trade_time(df)
    tt = pd.to_datetime(dfn["trade_time"], errors="coerce").dropna()
    if tt.empty:
        print("no valid trade_time")
        return {"source": name, "rows": int(len(df)), "valid_trade_time": 0}

    uniq_dates = sorted(tt.dt.date.unique().tolist())
    hours = tt.dt.hour.value_counts().sort_index()
    phase_top = {}
    if "trading_phase_code" in dfn.columns:
        phase_top = dfn["trading_phase_code"].fillna("NA").value_counts().head(10).to_dict()

    print("rows:", len(dfn))
    print("trade_time min/max:", tt.min(), tt.max())
    print("unique trade dates:", uniq_dates[:10], "... total", len(uniq_dates))
    print("hour dist:", hours.to_dict())
    if phase_top:
        print("phase top:", phase_top)
    print("head:")
    display(dfn.head(SHOW_HEAD))

    match_rows = None
    if expected_day is not None:
        match_rows = int((tt.dt.date == expected_day).sum())
        print(f"rows matching {expected_day}:", match_rows)

    return {
        "source": name,
        "rows": int(len(dfn)),
        "valid_trade_time": int(len(tt)),
        "min_trade_time": tt.min(),
        "max_trade_time": tt.max(),
        "unique_trade_dates": int(len(uniq_dates)),
        "rows_match_expected_day": match_rows,
    }


In [18]:
# NFS raw (直接源文件)
if nfs_raw_path.exists():
    nfs_df = pd.read_parquet(nfs_raw_path)
else:
    nfs_df = pd.DataFrame()
nfs_stat = summarize_snapshot(nfs_df, "NFS raw snapshot", expected_day=DATE)

# 本地 raw (sync 后落地)
if local_raw_path.exists():
    local_df = pd.read_parquet(local_raw_path)
else:
    local_df = pd.DataFrame()
local_stat = summarize_snapshot(local_df, "Local raw snapshot", expected_day=DATE)


===== NFS raw snapshot =====
rows: 1602330
trade_time min/max: 2026-02-06 09:25:00 2026-02-06 15:01:00
unique trade dates: [datetime.date(2026, 2, 6)] ... total 1
hour dist: {9: 213694, 10: 396024, 11: 204639, 12: 33720, 13: 375213, 14: 378305, 15: 735}
phase top: {'T': 833603, 'T0': 744875, 'B0': 18240, 'C0': 2190, 'P': 1346, 'T1': 948, 'E': 521, 'B1': 380, 'E0': 204, 'C1': 12}
head:


,code,trade_time,pre_close,last,open,high,low,close,volume,amount,...,ask_price4,ask_volume4,bid_price4,bid_volume4,ask_price5,ask_volume5,bid_price5,bid_volume5,iopv,trading_phase_code
0,123178.SZ,2026-02-06 09:25:00,143.2,140.559,140.559,140.559,140.559,0.0,890.0,125097.51,...,142.5,100.0,139.62,70.0,142.825,70.0,139.58,30.0,0.0,B0
1,123178.SZ,2026-02-06 09:26:30,143.2,140.559,140.559,140.559,140.559,0.0,890.0,125097.51,...,142.5,100.0,139.62,70.0,142.825,70.0,139.58,30.0,0.0,B0
2,123178.SZ,2026-02-06 09:27:30,143.2,140.559,140.559,140.559,140.559,0.0,890.0,125097.51,...,142.5,100.0,139.62,70.0,142.825,70.0,139.58,30.0,0.0,B0
3,123178.SZ,2026-02-06 09:28:30,143.2,140.559,140.559,140.559,140.559,0.0,890.0,125097.51,...,142.5,100.0,139.62,70.0,142.825,70.0,139.58,30.0,0.0,B0
4,123178.SZ,2026-02-06 09:29:30,143.2,140.559,140.559,140.559,140.559,0.0,890.0,125097.51,...,142.5,100.0,139.62,70.0,142.825,70.0,139.58,30.0,0.0,B0


rows matching 2026-02-06: 1602330
===== Local raw snapshot =====
rows: 1602330
trade_time min/max: 2026-02-06 09:25:00 2026-02-06 15:01:00
unique trade dates: [datetime.date(2026, 2, 6)] ... total 1
hour dist: {9: 213694, 10: 396024, 11: 204639, 12: 33720, 13: 375213, 14: 378305, 15: 735}
phase top: {'T': 833603, 'T0': 744875, 'B0': 18240, 'C0': 2190, 'P': 1346, 'T1': 948, 'E': 521, 'B1': 380, 'E0': 204, 'C1': 12}
head:


,code,trade_time,pre_close,last,open,high,low,close,volume,amount,...,ask_price4,ask_volume4,bid_price4,bid_volume4,ask_price5,ask_volume5,bid_price5,bid_volume5,iopv,trading_phase_code
0,123178.SZ,2026-02-06 09:25:00,143.2,140.559,140.559,140.559,140.559,0.0,890.0,125097.51,...,142.5,100.0,139.62,70.0,142.825,70.0,139.58,30.0,0.0,B0
1,123178.SZ,2026-02-06 09:26:30,143.2,140.559,140.559,140.559,140.559,0.0,890.0,125097.51,...,142.5,100.0,139.62,70.0,142.825,70.0,139.58,30.0,0.0,B0
2,123178.SZ,2026-02-06 09:27:30,143.2,140.559,140.559,140.559,140.559,0.0,890.0,125097.51,...,142.5,100.0,139.62,70.0,142.825,70.0,139.58,30.0,0.0,B0
3,123178.SZ,2026-02-06 09:28:30,143.2,140.559,140.559,140.559,140.559,0.0,890.0,125097.51,...,142.5,100.0,139.62,70.0,142.825,70.0,139.58,30.0,0.0,B0
4,123178.SZ,2026-02-06 09:29:30,143.2,140.559,140.559,140.559,140.559,0.0,890.0,125097.51,...,142.5,100.0,139.62,70.0,142.825,70.0,139.58,30.0,0.0,B0


rows matching 2026-02-06: 1602330


In [19]:
# Redis snapshot（按指定日期读）
redis_stat = {"source": "Redis snapshot", "rows": 0}
redis_df = pd.DataFrame()

try:
    rcfg = RedisConfig(
        host=str(live_cfg.get("redis_host", "")),
        port=int(live_cfg.get("redis_port", 6379)),
        db=int(live_cfg.get("redis_db", 0)),
        password=live_cfg.get("redis_password"),
        socket_timeout=live_cfg.get("redis_socket_timeout", 5),
    )
    client = SnapshotRedisClient(rcfg)
    asset_type = str(live_cfg.get("redis_asset_type", "cbond"))
    source = str(live_cfg.get("redis_source", "combiner"))
    stage = str(live_cfg.get("redis_stage", "raw"))

    symbols = client.list_symbols(source=source, stage=stage, asset_type=asset_type)
    symbols = symbols[:REDIS_SYMBOL_LIMIT] if REDIS_SYMBOL_LIMIT and REDIS_SYMBOL_LIMIT > 0 else symbols
    print("redis symbols used:", len(symbols))

    if REDIS_USE_RANGE:
        start_dt = pd.Timestamp.combine(DATE, pd.Timestamp("00:00").time())
        end_dt = pd.Timestamp.combine(DATE, pd.Timestamp("23:59:59").time())
        start_s = float(start_dt.timestamp())
        end_s = float(end_dt.timestamp())
        records = []
        for sym in symbols:
            rec = client.read_range(sym, source=source, stage=stage, asset_type=asset_type, start_time=start_s, end_time=end_s)
            if not rec:
                # 某些库可能用毫秒score
                rec = client.read_range(sym, source=source, stage=stage, asset_type=asset_type, start_time=start_s*1000, end_time=end_s*1000)
            if rec:
                for r in rec:
                    if "code" not in r:
                        r["code"] = sym
                records.extend(rec)
        redis_df = pd.DataFrame(records)
    else:
        redis_df = client.read_latest_df(symbols, source=source, stage=stage, asset_type=asset_type, limit=int(REDIS_LATEST_LIMIT))
        if not redis_df.empty:
            tmp = _normalize_trade_time(redis_df)
            tt = pd.to_datetime(tmp["trade_time"], errors="coerce")
            redis_df = tmp[tt.dt.date == DATE].copy()

    redis_stat = summarize_snapshot(redis_df, "Redis snapshot", expected_day=DATE)
except Exception as e:
    print("Redis read error:", e)


redis symbols used: 80
===== Redis snapshot =====
empty


In [20]:
# 三源汇总对比
cmp = pd.DataFrame([nfs_stat, local_stat, redis_stat])
display(cmp)


,source,rows,valid_trade_time,min_trade_time,max_trade_time,unique_trade_dates,rows_match_expected_day
0,NFS raw snapshot,1602330,1602330.0,2026-02-06 09:25:00,2026-02-06 15:01:00,1.0,1602330.0
1,Local raw snapshot,1602330,1602330.0,2026-02-06 09:25:00,2026-02-06 15:01:00,1.0,1602330.0
2,Redis snapshot,0,NaN,NaT,NaT,NaN,NaN


In [21]:
# DB 日频表：是否包含分钟级时间字段
db_tables = list(raw_cfg.get("db", {}).get("sync_tables", []))
minute_cols = {"trade_time", "timestamp", "ts"}

rows = []
for table in db_tables:
    df = read_table_range(paths_cfg["raw_data_root"], table, DATE, DATE)
    cols = set(df.columns) if not df.empty else set()
    has_minute_time_col = any(c in cols for c in minute_cols)
    rows.append({
        "table": table,
        "rows": int(len(df)),
        "has_trade_date": "trade_date" in cols,
        "has_minute_time_col": has_minute_time_col,
        "sample_cols": ", ".join(sorted(list(cols))[:12]),
    })

db_check = pd.DataFrame(rows)
display(db_check)
print("结论：DB这批 daily_* / trading_calendar 数据是日频，不是逐分钟快照。")


,table,rows,has_trade_date,has_minute_time_col,sample_cols
0,metadata.trading_calendar,0,False,False,
1,market_cbond.daily_price,625,True,False,"act_prev_close_price, amount, close_price, dea..."
2,market_cbond.daily_twap,373,True,False,"exchange_code, instrument_code, trade_date, tw..."
3,market_cbond.daily_vwap,373,True,False,"exchange_code, instrument_code, trade_date, up..."
4,market_cbond.daily_deriv,390,True,False,"base_rate, bond_prem_ratio, cb_call_price, cb_..."
5,market_cbond.daily_base,390,True,False,"base_rate, bond_prem_ratio, cb_act_prev_close_..."
6,market_cbond.daily_rating,1183,True,False,"exchange_code, id, instrument_code, rating, tr..."


结论：DB这批 daily_* / trading_calendar 数据是日频，不是逐分钟快照。
